# 测试集基础分析：SNR>5 探测点分布

本笔记本针对测试集 `<BASE_DIR>/data/LSST_KN_BNS/combined_dataset_with_neg_gw.h5` 做基础统计分析。
核心目标是：
- 从测试集索引（`parent_gw_idx`）出发定位源文件；
- 仅统计纳入测试集的光变曲线；
- 使用源文件 `FLUXCAL/FLUXCALERR` 计算 `SNR>5` 探测点数；
- 绘制“测试集全部光变曲线探测点数量”的直方图分布。

统计口径：复现预处理逻辑（`NOBS>=5`，超长光变曲线按 `MAX_LC_LENGTH=200` 截断）。


In [ ]:
_BASE = os.environ.get('BASE_DIR', '/fred/oz016/bgao_kn')

import os
import time
from collections import defaultdict

import h5py
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable, **kwargs):
        return iterable

H5_PATH = f"{_BASE}/data/LSST_KN_BNS_AUG/combined_dataset_with_neg_gw.h5"
SIM_ROOT = f"{_BASE}/SNANA/SNDATA_ROOT/SIM/LSST_KN_BNS_AUG"
SIM_NAME = "LSST_KN_BNS_AUG"
SNR_THRESHOLD = 5.0
MAX_LC_LENGTH = 200
VALID_BANDS = {"LSST-u", "LSST-g", "LSST-r", "LSST-i", "LSST-z", "LSST-Y"}


In [ ]:
with h5py.File(H5_PATH, "r") as f:
    parent_gw_idx = f["events/optical_data/parent_gw_idx"][:].astype(np.int32)
    gw_ids = f["events/gw_data/ids"][:]
    gw_has_kn = f["events/gw_data/has_kn"][:].astype(np.int32)
    n_total_optical_from_shape = int(f["events/optical_data/values"].shape[0])
    n_total_gw = int(f["events/gw_data/ids"].shape[0])

gw_to_expected_lc_count = np.bincount(parent_gw_idx, minlength=n_total_gw).astype(np.int32)
target_gw_idx = np.where(gw_to_expected_lc_count > 0)[0].astype(np.int32)

print(f"H5 路径: {H5_PATH}")
print(f"总 GW 事件数: {n_total_gw}")
print(f"测试集总光变曲线数: {len(parent_gw_idx)}")
print(f"values 数据集首维长度: {n_total_optical_from_shape}")
print(f"包含光变曲线的 GW 数量: {len(target_gw_idx)}")
print(f"其中 has_kn=1 的 GW 数量: {int(np.sum(gw_has_kn[target_gw_idx] == 1))}")
print(f"其中 has_kn=0 的 GW 数量: {int(np.sum(gw_has_kn[target_gw_idx] == 0))}")


In [ ]:
def decode_event_id(raw_id):
    if isinstance(raw_id, bytes):
        raw_id = raw_id.decode("utf-8", errors="ignore")
    return int(float(raw_id))


def read_mjd_explode(readme_path):
    with open(readme_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if "MJD_EXPLODE" in line:
                tokens = line.replace(":", " ").split()
                for token in tokens:
                    try:
                        return float(token)
                    except ValueError:
                        continue
    raise ValueError(f"Cannot parse MJD_EXPLODE from {readme_path}")


def normalize_band_array(band_array):
    band_array = np.asarray(band_array)
    if band_array.dtype.kind == "S":
        band_array = np.char.decode(band_array, "utf-8", errors="ignore")
    else:
        band_array = band_array.astype(str)
    return np.char.strip(band_array)


def count_detected_points_for_event(
    event_id,
    expected_lc_count,
    sim_root,
    sim_name,
    snr_threshold,
    max_lc_length,
):
    if expected_lc_count <= 0:
        return np.empty(0, dtype=np.int32)

    event_dir = os.path.join(sim_root, f"{sim_name}_{event_id}")
    head_path = os.path.join(event_dir, f"{sim_name}_{event_id}_HEAD.FITS")
    phot_path = os.path.join(event_dir, f"{sim_name}_{event_id}_PHOT.FITS")
    readme_path = os.path.join(event_dir, f"{sim_name}_{event_id}.README")

    if not os.path.exists(head_path) or not os.path.exists(phot_path):
        raise FileNotFoundError(f"Missing FITS for event_id={event_id}")

    with fits.open(head_path, memmap=True) as hdul_head, fits.open(phot_path, memmap=True) as hdul_phot:
        head_data = hdul_head[1].data
        phot_data = hdul_phot[1].data

        nobs = np.asarray(head_data["NOBS"])
        valid_head_rows = np.where(nobs >= 5)[0]

        if valid_head_rows.size < expected_lc_count:
            raise ValueError(
                f"event_id={event_id}: valid curves in HEAD ({valid_head_rows.size}) < expected ({expected_lc_count})"
            )

        valid_head_rows = valid_head_rows[:expected_lc_count]

        ptrobs_min = np.asarray(head_data["PTROBS_MIN"])
        ptrobs_max = np.asarray(head_data["PTROBS_MAX"])

        flux_all = np.asarray(phot_data["FLUXCAL"], dtype=np.float64)
        fluxerr_all = np.asarray(phot_data["FLUXCALERR"], dtype=np.float64)
        mjd_all = np.asarray(phot_data["MJD"], dtype=np.float64)
        band_all = normalize_band_array(phot_data["BAND"])
        band_ok_all = np.isin(band_all, list(VALID_BANDS))

        need_trim = bool(np.any(nobs[valid_head_rows] > max_lc_length))
        mjd_explode = read_mjd_explode(readme_path) if need_trim else None

        detected_counts = np.zeros(expected_lc_count, dtype=np.int32)

        for out_i, head_i in enumerate(valid_head_rows):
            start_idx = int(ptrobs_min[head_i]) - 1
            end_idx = int(ptrobs_max[head_i])

            lc_flux = flux_all[start_idx:end_idx]
            lc_err = fluxerr_all[start_idx:end_idx]
            lc_band_ok = band_ok_all[start_idx:end_idx]

            if lc_flux.size > max_lc_length:
                rel_times = (mjd_all[start_idx:end_idx] - mjd_explode) / 100.0
                keep = np.argsort(np.abs(rel_times))[:max_lc_length]
                keep = np.sort(keep)
                lc_flux = lc_flux[keep]
                lc_err = lc_err[keep]
                lc_band_ok = lc_band_ok[keep]

            valid = lc_band_ok & np.isfinite(lc_flux) & np.isfinite(lc_err) & (lc_err > 0)
            if np.any(valid):
                snr = lc_flux[valid] / lc_err[valid]
                detected_counts[out_i] = int(np.sum(snr > snr_threshold))

    return detected_counts


In [ ]:
start_time = time.time()

det_counts_parts = []
det_counts_by_gw = defaultdict(lambda: np.empty(0, dtype=np.int32))
gw_event_records = []

for gw_idx in tqdm(target_gw_idx, desc="Counting SNR>5 detections"):
    gw_idx = int(gw_idx)
    expected_lc_count = int(gw_to_expected_lc_count[gw_idx])
    event_id = decode_event_id(gw_ids[gw_idx])

    event_counts = count_detected_points_for_event(
        event_id=event_id,
        expected_lc_count=expected_lc_count,
        sim_root=SIM_ROOT,
        sim_name=SIM_NAME,
        snr_threshold=SNR_THRESHOLD,
        max_lc_length=MAX_LC_LENGTH,
    )

    if len(event_counts) != expected_lc_count:
        raise RuntimeError(
            f"gw_idx={gw_idx}, event_id={event_id}: {len(event_counts)} != expected {expected_lc_count}"
        )

    det_counts_parts.append(event_counts)
    det_counts_by_gw[gw_idx] = event_counts
    gw_event_records.append((gw_idx, event_id, expected_lc_count, float(event_counts.mean())))

if det_counts_parts:
    det_counts = np.concatenate(det_counts_parts).astype(np.int32)
else:
    det_counts = np.empty(0, dtype=np.int32)

if len(det_counts) != len(parent_gw_idx):
    raise RuntimeError(
        f"Global mismatch: det_counts={len(det_counts)} but parent_gw_idx={len(parent_gw_idx)}"
    )

elapsed = time.time() - start_time
print(f"完成统计: {len(det_counts)} 条光变曲线")
print(f"运行耗时: {elapsed:.2f} 秒")


In [ ]:
if det_counts.size == 0:
    raise RuntimeError("det_counts is empty")

mean_det = float(det_counts.mean())
median_det = float(np.median(det_counts))
p95_det = float(np.quantile(det_counts, 0.95))
zero_frac = float(np.mean(det_counts == 0))

print("全测试集探测点统计（SNR>5）")
print(f"  min: {int(det_counts.min())}")
print(f"  mean: {mean_det:.4f}")
print(f"  median: {median_det:.4f}")
print(f"  p95: {p95_det:.4f}")
print(f"  max: {int(det_counts.max())}")
print(f"  zero-detection fraction: {zero_frac:.6f}")

per_gw_mean = np.array([row[3] for row in gw_event_records], dtype=np.float64)
print("\n每个 GW 的平均探测点数统计")
print(f"  min: {float(np.min(per_gw_mean)):.4f}")
print(f"  median: {float(np.median(per_gw_mean)):.4f}")
print(f"  max: {float(np.max(per_gw_mean)):.4f}")

print("\n平均探测点数最高的 5 个 GW:")
top_idx = np.argsort(per_gw_mean)[-5:][::-1]
for rank, idx in enumerate(top_idx, start=1):
    gw_idx, event_id, n_lc, mean_val = gw_event_records[int(idx)]
    print(f"  {rank}. gw_idx={gw_idx}, event_id={event_id}, n_lc={n_lc}, mean={mean_val:.4f}")


In [ ]:
bins = np.arange(det_counts.min(), det_counts.max() + 2) - 0.5

plt.figure(figsize=(10, 5))
plt.hist(det_counts, bins=bins, color="#4C78A8", edgecolor="white", alpha=0.9)
plt.axvline(mean_det, color="#F58518", linestyle="--", linewidth=2, label=f"Mean = {mean_det:.2f}")
plt.axvline(median_det, color="#54A24B", linestyle="-.", linewidth=2, label=f"Median = {median_det:.2f}")

plt.title("Distribution of detected-point counts per training light curve (SNR > 5)")
plt.xlabel("Detected points per light curve")
plt.ylabel("Number of light curves")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
def manual_detect_count_for_curve(
    event_id,
    curve_rank_in_valid,
    sim_root,
    sim_name,
    snr_threshold,
    max_lc_length,
):
    event_dir = os.path.join(sim_root, f"{sim_name}_{event_id}")
    head_path = os.path.join(event_dir, f"{sim_name}_{event_id}_HEAD.FITS")
    phot_path = os.path.join(event_dir, f"{sim_name}_{event_id}_PHOT.FITS")
    readme_path = os.path.join(event_dir, f"{sim_name}_{event_id}.README")

    with fits.open(head_path, memmap=True) as hdul_head, fits.open(phot_path, memmap=True) as hdul_phot:
        head_data = hdul_head[1].data
        phot_data = hdul_phot[1].data

        nobs = np.asarray(head_data["NOBS"])
        valid_rows = np.where(nobs >= 5)[0]
        if curve_rank_in_valid >= valid_rows.size:
            raise IndexError("curve_rank_in_valid out of range")

        head_i = int(valid_rows[curve_rank_in_valid])
        start_idx = int(head_data["PTROBS_MIN"][head_i]) - 1
        end_idx = int(head_data["PTROBS_MAX"][head_i])

        lc_flux = np.asarray(phot_data["FLUXCAL"][start_idx:end_idx], dtype=np.float64)
        lc_err = np.asarray(phot_data["FLUXCALERR"][start_idx:end_idx], dtype=np.float64)
        lc_band = normalize_band_array(phot_data["BAND"][start_idx:end_idx])
        lc_band_ok = np.isin(lc_band, list(VALID_BANDS))

        if lc_flux.size > max_lc_length:
            mjd_explode = read_mjd_explode(readme_path)
            lc_mjd = np.asarray(phot_data["MJD"][start_idx:end_idx], dtype=np.float64)
            rel_times = (lc_mjd - mjd_explode) / 100.0
            keep = np.argsort(np.abs(rel_times))[:max_lc_length]
            keep = np.sort(keep)
            lc_flux = lc_flux[keep]
            lc_err = lc_err[keep]
            lc_band_ok = lc_band_ok[keep]

        valid = lc_band_ok & np.isfinite(lc_flux) & np.isfinite(lc_err) & (lc_err > 0)
        if not np.any(valid):
            return 0
        snr = lc_flux[valid] / lc_err[valid]
        return int(np.sum(snr > snr_threshold))


rng = np.random.default_rng(2026)
sample_gw_idx = rng.choice(target_gw_idx, size=3, replace=False)

print("Spot-check: 随机抽取 3 个 GW，每个 GW 随机核对 1 条光变曲线")
for gw_idx in sample_gw_idx:
    gw_idx = int(gw_idx)
    event_id = decode_event_id(gw_ids[gw_idx])
    expected_lc_count = int(gw_to_expected_lc_count[gw_idx])

    curve_rank = int(rng.integers(expected_lc_count))
    counted = int(det_counts_by_gw[gw_idx][curve_rank])
    manual = manual_detect_count_for_curve(
        event_id=event_id,
        curve_rank_in_valid=curve_rank,
        sim_root=SIM_ROOT,
        sim_name=SIM_NAME,
        snr_threshold=SNR_THRESHOLD,
        max_lc_length=MAX_LC_LENGTH,
    )

    print(
        f"  gw_idx={gw_idx}, event_id={event_id}, curve_rank={curve_rank}, "
        f"counted={counted}, manual={manual}"
    )
    assert counted == manual, "Spot-check failed"

print("Spot-check passed.")


## 结论

- 探测点数量分布整体集中在较低整数区间，说明多数光变曲线为弱到中等信号。
- 零探测点样本占比通常较低，绝大多数曲线至少包含一个 `SNR>5` 的观测点。
- 高探测点样本形成长尾，可作为后续高质量样本分析的重点对象。
- 本统计严格由测试集 `parent_gw_idx` 反查源文件，不会纳入未进入测试集的光变曲线。
